In [1]:
from pathlib import Path

Path.cwd()


WindowsPath('D:/data-engineering-pipeline/notebooks')

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT


WindowsPath('D:/data-engineering-pipeline')

In [3]:
import sys

project_root = str(PROJECT_ROOT)

if project_root not in sys.path:
    sys.path.insert(0, project_root)


In [4]:
from src.spark_session import get_spark_session

In [5]:
spark = get_spark_session()

In [6]:
spark.version

'4.2.0'

In [7]:
input_file = PROJECT_ROOT / "data" / "raw" / "train.csv"

input_file.exists()

True

In [8]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(input_file))
)

In [9]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [10]:
df.show(5, truncate=False)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id       |vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|pickup_longitude  |pickup_latitude   |dropoff_longitude |dropoff_latitude  |store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|2        |2016-03-14 17:24:55|2016-03-14 17:32:30|1              |-73.9821548461914 |40.76793670654297 |-73.96463012695312|40.765602111816406|N                 |455          |
|id2377394|1        |2016-06-12 00:43:35|2016-06-12 00:54:38|1              |-73.98041534423828|40.738563537597656|-73.99948120117188|40.73115158081055 |N                 |663          |
|id3858529|2        |2016-01-19 11:35:24|2016-01-19 12:10:48|1   

In [11]:
df.count()

1458644

In [12]:
df.columns

['id',
 'vendor_id',
 'pickup_datetime',
 'dropoff_datetime',
 'passenger_count',
 'pickup_longitude',
 'pickup_latitude',
 'dropoff_longitude',
 'dropoff_latitude',
 'store_and_fwd_flag',
 'trip_duration']

In [13]:
df.select("id").count(), df.select("id").distinct().count()

(1458644, 1458644)

In [14]:
from pyspark.sql.functions import col, sum

null_counts = df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)

null_counts.show()

+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
| id|vendor_id|pickup_datetime|dropoff_datetime|passenger_count|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|store_and_fwd_flag|trip_duration|
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
|  0|        0|              0|               0|              0|               0|              0|                0|               0|                 0|            0|
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+



In [15]:
df.groupBy("passenger_count") \
    .count() \
    .orderBy("passenger_count") \
    .show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|              0|     60|
|              1|1033540|
|              2| 210318|
|              3|  59896|
|              4|  28404|
|              5|  78088|
|              6|  48333|
|              7|      3|
|              8|      1|
|              9|      1|
+---------------+-------+



In [16]:
df.select("trip_duration").describe().show()

+-------+-----------------+
|summary|    trip_duration|
+-------+-----------------+
|  count|          1458644|
|   mean|959.4922729603659|
| stddev|5237.431724497624|
|    min|                1|
|    max|          3526282|
+-------+-----------------+



In [17]:
df.filter(col("trip_duration") <= 0).count()

0

In [19]:
df.orderBy(
    col("trip_duration").desc()
).select(
    "id",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration"
).show(20, truncate=False)

+---------+-------------------+-------------------+-------------+
|id       |pickup_datetime    |dropoff_datetime   |trip_duration|
+---------+-------------------+-------------------+-------------+
|id0053347|2016-02-13 22:46:52|2016-03-25 18:18:14|3526282      |
|id1325766|2016-01-05 06:14:15|2016-01-31 01:01:07|2227612      |
|id0369307|2016-02-13 22:38:00|2016-03-08 15:57:38|2049578      |
|id1864733|2016-01-05 00:19:42|2016-01-27 11:08:38|1939736      |
|id1942836|2016-02-15 23:18:06|2016-02-16 23:17:58|86392        |
|id0593332|2016-05-31 13:00:39|2016-06-01 13:00:30|86391        |
|id0953667|2016-05-06 00:00:10|2016-05-07 00:00:00|86390        |
|id2837671|2016-06-30 16:37:52|2016-07-01 16:37:39|86387        |
|id1358458|2016-06-23 16:01:45|2016-06-24 16:01:30|86385        |
|id2589925|2016-05-17 22:22:56|2016-05-18 22:22:35|86379        |
|id3346235|2016-05-09 15:59:04|2016-05-10 15:58:42|86378        |
|id3782820|2016-05-12 13:48:19|2016-05-13 13:47:57|86378        |
|id1565504